# VisDSR v2: InternVL3.5-8B-HF on Kaggle

Run the cells from top to bottom with **GPU** and **Internet** enabled. The default stage makes only three smoke calls and exports a backup automatically. For the first smoke, attach `visdsr_v2_diagnostics_complete.zip` as a private Dataset before setup. It contains the reviewed Qwen smoke and eight diagnostic records. For calibration or resume, attach the newest combined v2 export instead. Attach only one VisDSR results input; earlier v1 ZIPs are rejected.

The study uses 8/16 elements and 1/4 operations. Its protocol is in `STUDY_V2.md`. Smoke results must be reviewed before calibration; main requires a frozen protocol and a separate review. Every task-condition is independent. Inference runs on the free GPU; no paid API is used.

To resume later, download `visdsr_v2_results.zip`, upload that latest ZIP as a private Kaggle Dataset, and attach it to this notebook. Kaggle may expand the ZIP; both forms are supported. Run this notebook with one model at a time.


In [ ]:
MODEL = "model2"  # Fixed model family for this notebook
STAGE = "smoke"  # smoke, calibration, or main
SMOKE_REVIEWED = False
MAIN_APPROVED = False
NEW_CALLS_PER_CHUNK = 20
REPOSITORY = "https://github.com/Abhishekgit01/VisDSR.git"
BRANCH = "main"

if STAGE not in ("smoke", "calibration", "main"):
    raise ValueError("STAGE must be smoke, calibration, or main")
if STAGE != "smoke" and not SMOKE_REVIEWED:
    raise RuntimeError("Review this model's v2 smoke before selecting calibration or main")
if STAGE == "main" and not MAIN_APPROVED:
    raise RuntimeError("Review calibration and freeze the protocol before selecting main")
if type(NEW_CALLS_PER_CHUNK) is not int or NEW_CALLS_PER_CHUNK < 1:
    raise ValueError("NEW_CALLS_PER_CHUNK must be a positive integer")
print("Selected stage:", STAGE, "Model:", MODEL)


## 1. Load the project and restore a backup

A new v2 project folder is used at `/kaggle/working/VisDSR_v2`. The setup verifies the backup's code, task, and file hashes before copying results. A project copy attached as an input is supported when cloning is unavailable. Internet is also needed for packages and model weights.


In [ ]:
from pathlib import Path
import os
import shutil
import socket
import subprocess
import sys

WORK = Path("/kaggle/working")
if not WORK.exists():
    raise RuntimeError("This execution notebook requires a free Kaggle GPU session")
ROOT = WORK / "VisDSR_v2"
INPUT = Path("/kaggle/input")
EXPORT = WORK / "visdsr_v2_results.zip"


def run_command(*parts, cwd=None):
    subprocess.run(parts, cwd=cwd or ROOT, check=True)


if not (ROOT / "configs/experiment.yaml").exists():
    candidates = [p.parent for p in INPUT.rglob("STUDY_V2.md")
                  if (p.parent / "sim/dsu.cpp").exists()] if INPUT.exists() else []
    if len(candidates) > 1:
        raise RuntimeError("Attach only one v2 project copy")
    if candidates:
        shutil.copytree(candidates[0], ROOT, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns(".git", "__pycache__", "data", "results"))
    else:
        if not REPOSITORY.startswith("https://github.com/") or any(c in REPOSITORY for c in "[]()"):
            raise ValueError("REPOSITORY must be a plain GitHub URL")
        try:
            socket.getaddrinfo("github.com", 443)
        except OSError as exc:
            raise RuntimeError("Turn Internet ON in Kaggle Session options before running setup") from exc
        run_command("git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPOSITORY, str(ROOT), cwd=WORK)

run_command(sys.executable, "-m", "pip", "install", "-q", "-r",
            str(ROOT / "environment/requirements.txt"))
import yaml
cfg = yaml.safe_load((ROOT / "configs/experiment.yaml").read_text())
if cfg.get("protocol_id") != "visdsr-dsu-v2":
    raise RuntimeError("This project folder is not v2; use a fresh session with the revised notebook")
settings = next(item for item in cfg["models"] if item["name"] == MODEL)
print("Project:", ROOT, "Protocol:", cfg["protocol_id"], "Stage:", STAGE)
print("Model:", settings["id"], "Revision:", settings["revision"])
if (ROOT / ".git").exists():
    run_command("git", "rev-parse", "HEAD")

if list((ROOT / "results/cache").glob("*.json")):
    print("Using this session's existing cache; the attached backup will not overwrite it")
else:
    restore = [sys.executable, "-m", "study_transfer", "--restore-auto", str(INPUT)]
    if MODEL == "model2":
        restore.append("--require-backup")
    run_command(*restore)


def export_results():
    if not (ROOT / "data/pilot2/tasks.jsonl").exists():
        print("No v2 calibration data is ready to export yet")
        return
    run_command(sys.executable, "-m", "manifest")
    run_command(sys.executable, "-m", "study_transfer", "--export", str(EXPORT))
    print("BACKUP READY:", EXPORT)
    print("Download visdsr_v2_results.zip from Kaggle's Output panel before leaving this session")


## 2. Install inference packages and check the GPU

The versions below were recorded in the successful v1 InternVL GPU session. Keep Kaggle's installed PyTorch/CUDA build. The runner logs the actual environment and pinned model revision with the output.


In [ ]:
import platform
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
import torch
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Select a free GPU accelerator in Kaggle Session options")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name}; VRAM {props.total_memory / 2**30:.2f} GiB")
run_command(sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed",
            "-r", str(ROOT / "environment/requirements-inference.txt"))
print("Quantization:", settings["quantization"])
print("Generation:", {name: settings[name] for name in
                      ("do_sample", "num_beams", "max_output_tokens")})


## 3. Build and validate tasks and images

Calibration has three tasks in each of the four main difficulty cells. The same source PNG bytes are reused for both models. This step generates data and does not call a model.


In [ ]:
fonts = [Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
         Path("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf")]
if shutil.which("dot") is None or not all(path.exists() for path in fonts):
    run_command("apt-get", "update", "-qq")
    run_command("apt-get", "install", "-y", "-qq", "graphviz", "fonts-dejavu-core")
run_command("make", "build")
input_split = "main" if STAGE == "main" else "pilot2"
if STAGE == "main" and not (ROOT / "FREEZE.json").exists():
    raise RuntimeError("Main requires the reviewed frozen protocol in this checkout")
if not (ROOT / "data" / input_split / "tasks.jsonl").exists():
    run_command(sys.executable, "-m", "gen.generate", "--split", input_split)
if not (ROOT / "data" / input_split / "manifest.csv").exists():
    run_command(sys.executable, "-m", "gen.render", "--split", input_split)
run_command(sys.executable, "-m", "gen.validate", "--split", input_split)
print("Validated input:", input_split)
run_command(sys.executable, "-m", "eval.run", "--split", input_split,
            "--model", MODEL, "--dry-run")


## 4. Run the selected stage

With `STAGE = "smoke"`, this cell runs exactly three conditions on an 8-element four-operation task and stops. Every new response is saved immediately. It exports the ZIP after a completed run or a normal cell interruption.

After smoke review, calibration runs at most `NEW_CALLS_PER_CHUNK` new calls per execution. It reuses smoke responses and other matching cached calls. Rerun this cell for the next chunk. Main follows the same resume behavior after its separate freeze/review.

When changing stages, rerun the configuration and task-validation cells before this cell.


In [ ]:
import csv
input_split = "main" if STAGE == "main" else "pilot2"
try:
    if STAGE == "smoke":
        print("SMOKE: exactly one task in T-dir, R-dir, and G-dir")
        run_command(sys.executable, "-m", "eval.run", "--split", "pilot2",
                    "--model", MODEL, "--smoke")
        print("STOP: share and review this smoke report and backup before enabling calibration")
    else:
        if not SMOKE_REVIEWED:
            raise RuntimeError("Review this model's v2 smoke, then set SMOKE_REVIEWED = True")
        if STAGE == "main" and not MAIN_APPROVED:
            raise RuntimeError("Main requires the reviewed calibration and MAIN_APPROVED = True")
        run_command(sys.executable, "-m", "eval.run", "--split", input_split,
                    "--model", MODEL, "--after-smoke-review",
                    "--max-new-calls", str(NEW_CALLS_PER_CHUNK))
        score_name = f"scores_{MODEL}.csv" if STAGE == "main" else f"scores_pilot2_{MODEL}.csv"
        with (ROOT / "results" / score_name).open(newline="") as source:
            rows = list(csv.DictReader(source))
        total = 400 if STAGE == "main" else 60
        print(f"PROGRESS: {len(rows)}/{total} cached and scored responses for {MODEL}")
        if len(rows) == total:
            run_command(sys.executable, "-m", "analysis.analyze", "--split", input_split,
                        "--models", MODEL)
            print("This model's selected stage is complete; download the backup for review")
        else:
            print("Download the backup now. Run this cell again to make the next bounded chunk")
finally:
    export_results()


## 5. Export and download a backup

The previous cell exports automatically. Run this cell again after a partial run if needed. In Kaggle's **Output** panel, find `/kaggle/working/visdsr_v2_results.zip`, refresh the list, and use its download control. Download before leaving the session. Preserve the latest version outside Kaggle's temporary session and attach it when resuming.


In [ ]:
export_results()
